
# PLL Noise Shaping Simulation for https://ieeexplore.ieee.org/document/8686112

This notebook models and visualizes the **noise shaping** behavior of an All-Digital Phase-Locked Loop (ADPLL) using control theory and frequency response analysis.

It computes transfer functions for:
- **DCO** (Digitally Controlled Oscillator)  
- **DLF** (Digital Loop Filter)  
- **Reference path** (REF)  
- **Noise Transfer Functions** (NTFs) for different noise sources  

Finally, it generates **Bode plots** to visualize how each block shapes the overall PLL noise.

NOTE: I deeply apologize if I am butchering the terminology of PLL, signal processing, or filter design. Not my expertise.

In [ ]:

# Libraries needed
import numpy as np
import matplotlib.pyplot as plt
! mkdir -p outputs

## 1. PLL and Simulation Parameters
Define reference frequency, multiplication ratio, and
key constants describing DCO behavior and loop filter.

In [ ]:
F_REF = 100e6      # Reference frequency (100 MHz)
N = 10             # PLL multiplication factor (F_VCO = N * F_REF)
zigma_t = 0.4e-12  # BBPD equivalent RMS jitter (0.4 ps)
Tr = 8.226e-11     # Delay of the SSBBPD
fc_hz = 1e6        # Loop filter cutoff frequency (10 MHz)
F_VCO = N * F_REF  # Output frequency of the DCO
inj_enable = False # Enable the injection locking
fine_bits = 5      # Fine quantization bits
DLF_bits = 16      # Digital loop filter quantization bits (must be larger than `fine_bits`)
version = "paper"  # Assign it to "paper", "tsmcn65"

if version == "paper":
    # For paper (Run step 7 with disabled simulations to get this)
    Kvco_noise_f2 = 29.645203734469376  # K_WA
    Kvco_noise_f3 = 19775728.672144137  # Kfkr_WA
elif version == "tsmcn65":
# For 65nm real simulations (Run step 7 with enabled simulations to get this)
    Kvco_noise_f2 = 103.10721986896505  # K_WA
    Kvco_noise_f3 = 476291161.0944519   # Kfkr_WA
elif version == "tsmcn28":
    Kvco_noise_f2 = 103.10721986896505  # K_WA
    Kvco_noise_f3 = 476291161.0944519   # Kfkr_WA

## 2. DCO Gain (K_DCO)
The DCO gain defines how much the output frequency changes
with respect to a digital control word (e.g., fine delay bits).

Here we compute the *effective* K_DCO from a differential
delay between two close delay cell configurations.

Note however that the difference is taken from the fine cells
and not the mid or coarse.

In [ ]:
K_DCO = np.abs(1 / (1 / F_VCO + 2.827e-11) - 1 / (1 / F_VCO + 2.812e-11)) / (1 << 5)  # From simulations
#K_DCO = 6.664e+04
print(f"K_DCO = {K_DCO:.3e} Hz/LSB")

## 3. Loop Filter and Phase Detector Parameters
The Digital Loop Filter (DLF) is modeled as:
  H_DLF(s) = Kp + Ki/s = (Kp * s + Ki) / s

The BBPD (Bang-Bang Phase Detector) gain and reference noise
parameters are also defined here.

In [ ]:
RC = 1/(2*np.pi*fc_hz)
R = 1   # Note: This is kinda arbitrary
C = RC / R

Kp = R - (1/F_REF)/(2*C)
Ki = (1/F_REF)/C
print(f"Kp = {Kp}, Ki = {Ki}")

FIXED_BITS = (DLF_bits - fine_bits + 1)  # Bits from right where the fixed point exists.
Kpb = int((1 << FIXED_BITS) * Kp)
Kib = int((1 << FIXED_BITS) * Ki)
print(f"Kpb = {Kpb}, Kib = {Kib}")

K_BBPD = np.sqrt(2 / np.pi) * 1 / (2 * np.pi * F_REF * zigma_t)

assert Kpb < (1 << DLF_bits) and Kib < (1 << DLF_bits), f"Number of bits are not enough. Check Kpb and Kib"

## 4. Reference Noise Shaping
There is a *magical* function that models the phase noise of the input 
buffer of the reference. Their parameters are written here

In [ ]:
Temp = 300  # Temperature (in kelvin)
k_const = 1.38e-23  # Boltzman constant
n_ref_noise = np.power(10.0, -155.0 / 10)  # White noise intensity
I_BUF_N = 0.0020  # Reference buffer bias current [A]

# 5. Frequency Axis Setup
We'll analyze system behavior across 1 kHz to 100 MHz.

In [ ]:
freqs = np.logspace(4, 7.8, 500)
omega = 2j * np.pi * freqs

## 6. Reference Path Noise Model
The magic function that models the buffer for the reference.
Models the thermal noise contribution of the reference buffer.
The function REF(x) approximates the PSD shaping over frequency.

In [ ]:
v_vdd = 1.0
v_swing = v_vdd - 0.4
k_ref = ((2 * k_const * Temp) / I_BUF_N) * (((4 / 3) / v_swing) + (1 / v_vdd))
ffrac = F_REF

def REF_old(x):
    f = np.imag(x) / (2*np.pi)
    return k_ref * (ffrac / f) + n_ref_noise

REF_old_resp = REF_old(omega)
REF_old_mag = np.abs(REF_old_resp)
REF_old_phase = np.angle(REF_old_resp, deg=True)

gain_ref = 4.8e-8
def REF(x):
    return gain_ref * (x + 2*np.pi*400e3) / ((x) * (x + 2*np.pi*9e6))  #

REF_resp = REF(omega)
REF_mag = np.abs(REF_resp)
REF_phase = np.angle(REF_resp, deg=True)

## 7. DCO Path Noise Model

This is another magic function that models the ring oscillator for the VCO.

In [ ]:
def DCO(x):
    f = np.imag(x) / (2*np.pi)
    return Kvco_noise_f2 / np.power(f, 2) + Kvco_noise_f3 / np.power(f, 3)

DCO_resp = DCO(omega)
DCO_mag = np.abs(DCO_resp)
DCO_phase = np.angle(DCO_resp, deg=True)

## 8. SSBBPD Path Noise Model

This is the last of the magic functions. Is extracted from the paper.

In [ ]:
def SSBBPD(x):
    f = np.imag(x) / (2*np.pi)
    return (1-2/np.pi)*2/F_REF*(np.sinc(f/F_REF) ** 2)

SSBBPD_resp = SSBBPD(omega)
SSBBPD_mag = np.abs(SSBBPD_resp)
SSBBPD_phase = np.angle(SSBBPD_resp, deg=True)

SSBPD_gain = (1-2/np.pi)*2/F_REF
print(f"SSBPD_gain = {SSBPD_gain}, {10*np.log10(SSBPD_gain)}db")

## 9. System Components

We model the DCO and DLF as transfer functions.

In [ ]:
# DLF(z): Kp + Ki / (1 - z^-1)
# analog counterpart (Bilinear transform, and the sampling time is the reference clock)
# DLF(s) = (Kp + Ki/2)*(s + Ki / ((Kp + Ki/2)*T)) / s
#H_DLF_resp = (Kp + Ki/2)*(omega + Ki / ((Kp + Ki/2)/F_REF)) / omega
H_DLF_resp = R*(omega + 1/RC) / omega
H_DLF_mag = np.abs(H_DLF_resp)
H_DLF_phase = np.angle(H_DLF_resp, deg=True)

# DCO: K_DCO / s
H_DCO_resp = K_DCO / omega

## 10. Other Transfer Functions (Loop Dynamics)

H_RL and H_UP capture the timing response of sampling-based
feedback paths such as subsampling BBPDs and the injection loop.
Is based in another paper (https://ieeexplore.ieee.org/document/1088109), with a `beta=1`.

In [ ]:
def H_RL(x):
    if inj_enable:
        return 1 - np.exp(-x * Tr / 2) * np.sinc(np.imag(x) * Tr / 2)
    else:
        return np.ones((len(x),))

H_RL_resp = H_RL(omega)
H_RL_mag = np.abs(H_RL_resp)
H_RL_phase = np.angle(H_RL_resp, deg=True)

def H_UP(x):
    if inj_enable:
        return N * np.exp(-x * Tr / 2) * np.sinc(np.imag(x) * Tr / 2)
    else:
        return np.zeros((len(x),))

H_UP_resp = H_UP(omega)
H_UP_mag = np.abs(H_UP_resp)
H_UP_phase = np.angle(H_UP_resp, deg=True)

## 11. Open-Loop Response (T)

T(s) models the open-loop transfer function of the PLL:

   T(s) = e^(-s*Tr) * (1/N) * K_BBPD * H_DLF(s) * H_DCO(s) * H_RL(s)

In [ ]:
TN_resp = np.exp(-omega * Tr) * (1 / N) * K_BBPD * H_DLF_resp * H_DCO_resp
TN_mag = np.abs(TN_resp)
TN_phase = np.angle(TN_resp, deg=True)

T_resp = TN_resp * H_RL_resp
T_mag = np.abs(T_resp)
T_phase = np.angle(T_resp, deg=True)

## 12. Noise Transfer Functions (NTFs)

According to the paper, these are the noise contributions of each component:

   - NTF_REF: Reference input noise
   - NTF_BBPD: Phase detector quantization noise
   - NTF_DCO: DCO intrinsic noise

NOTE: Not even me understand these correctly. We assume is just whatever
the original author drawed in his version of the phase noise.

In [ ]:
NTF_REF_resp = H_UP_resp + T_resp * np.exp(omega * Tr) / (1 + T_resp)
NTF_REF_mag = np.abs(NTF_REF_resp)
NTF_REF_phase = np.angle(NTF_REF_resp, deg=True)
S_REF_resp = REF_resp * (np.abs(NTF_REF_resp) ** 2)  # TODO: Why the magnitude squared?
S_REF_mag = np.abs(S_REF_resp)

NTFN_BBPD_resp = H_DLF_resp * H_DCO_resp / (1 + TN_resp)
NTFN_BBPD_mag = np.abs(NTFN_BBPD_resp)
NTFN_BBPD_phase = np.angle(NTFN_BBPD_resp, deg=True)
NTF_BBPD_resp = H_RL_resp * H_DLF_resp * H_DCO_resp / (1 + T_resp)
NTF_BBPD_mag = np.abs(NTF_BBPD_resp)
NTF_BBPD_phase = np.angle(NTF_BBPD_resp, deg=True)
S_BBPD_resp = SSBBPD_resp * (np.abs(NTF_BBPD_resp) ** 2)  # TODO: Why the magnitude squared?
S_BBPD_mag = np.abs(S_BBPD_resp)

NTFN_DCO_resp = 1 / (1 + TN_resp)
NTFN_DCO_mag = np.abs(NTFN_DCO_resp)
NTFN_DCO_phase = np.angle(NTFN_DCO_resp, deg=True)
NTF_DCO_resp = H_RL_resp / (1 + T_resp)
NTF_DCO_mag = np.abs(NTF_DCO_resp)
NTF_DCO_phase = np.angle(NTF_DCO_resp, deg=True)
S_DCO_resp = DCO_resp * (np.abs(NTF_DCO_resp) ** 2)  # TODO: Why the magnitude squared?
S_DCO_mag = np.abs(S_DCO_resp)

## 13. Closed loop PLL transfer function

Combine all system blocks to observe the overall PLL frequency
response and its noise shaping performance.

In [ ]:
PLL_resp = S_REF_resp + S_BBPD_resp + S_DCO_resp
PLL_mag = np.abs(PLL_resp)
PLL_phase = np.angle(PLL_resp, deg=True)

## 14. Visualization

Plot magnitude (dB) and phase (rad) for each subsystem and
noise transfer function to understand the shaping behavior.

In [ ]:
plt.figure(figsize=(9, 6))

# Magnitude
#plt.subplot(2, 1, 1)
plt.semilogx(freqs, 10 * np.log10(DCO_mag), color="#9C9C9C", label="DCO")
plt.semilogx(freqs, 10 * np.log10(REF_mag), color="#07580E", label="REF")
plt.semilogx(freqs, 10 * np.log10(S_REF_mag), color="#FF0000", label="REF (Shaped)")
plt.semilogx(freqs, 10 * np.log10(S_BBPD_mag), color="#929408", label="BBPD (Shaped)")
plt.semilogx(freqs, 10 * np.log10(S_DCO_mag), color="#0000FF", label="DCO (Shaped)")
plt.semilogx(freqs, 10 * np.log10(PLL_mag), color="#000000", label="PLL")
plt.title('PLL Noise Shaping – Magnitude')
plt.ylabel('Magnitude (dB)')
plt.grid(True, which="both", ls="-")
plt.legend(fontsize=8, loc='lower left')

# Phase
#plt.subplot(2, 1, 2)
#plt.semilogx(freqs, DCO_phase, color="#9C9C9C", label="DCO")
#plt.semilogx(freqs, REF_phase, color="#07580E", label="REF")
#plt.semilogx(freqs, NTF_REF_phase, color="#FF0000", label="$NTF_{REF}$")
#plt.semilogx(freqs, NTF_BBPD_phase, color="#929408", label="$NTF_{BBPD}$")
#plt.semilogx(freqs, NTF_DCO_phase, color="#0000FF", label="$NTF_{DCO}$")
#plt.semilogx(freqs, PLL_phase, color="#000000", label="PLL")
#plt.ylabel('Phase (rad)')
#plt.grid(True, which="both", ls="-")
#plt.legend(fontsize=8, loc='lower left')

plt.xlabel('Frequency (Hz)')
plt.tight_layout()
plt.savefig("outputs/pll_pn.pdf")

# For debug
plt.figure(figsize=(9, 12))
plt.subplot(2, 1, 1)
plt.semilogx(freqs, 10 * np.log10(H_DLF_mag), color="#000000", label="DLF")
plt.semilogx(freqs, 10 * np.log10(T_mag), color="#9C9C9C", label="T")
plt.semilogx(freqs, 10 * np.log10(TN_mag), color="#707070", label="TN")
plt.semilogx(freqs, 10 * np.log10(H_RL_mag), color="#FF00DD", label="$H_{RL}$")
plt.semilogx(freqs, 10 * np.log10(NTF_REF_mag), color="#FF0000", label="$NTF_{REF}$")
plt.semilogx(freqs, 10 * np.log10(NTFN_BBPD_mag), color="#CCCF02", label="$NTFN_{BBPD}$")
plt.semilogx(freqs, 10 * np.log10(NTF_BBPD_mag), color="#929408", label="$NTF_{BBPD}$")
plt.semilogx(freqs, 10 * np.log10(NTFN_DCO_mag), color="#6F72FF", label="$NTFN_{DCO}$")
plt.semilogx(freqs, 10 * np.log10(NTF_DCO_mag), color="#0000FF", label="$NTF_{DCO}$")
plt.semilogx(freqs, 10 * np.log10(SSBBPD_mag), color="#00FF2A", label="$Q_{BBPD}$")
# plt.semilogx(freqs, 10 * np.log10(REF_old_mag), color="#0000FF", label="REF (old)")
# plt.semilogx(freqs, 10 * np.log10(REF_mag), color="#FF0000", label="REF")
plt.title('Filter Shaping – Magnitude')
plt.ylabel('Magnitude')
plt.grid(True, which="both", ls="-")
plt.legend(fontsize=8, loc='lower left')
plt.xlabel('Frequency (Hz)')

plt.subplot(2, 1, 2)
plt.semilogx(freqs, H_DLF_phase, color="#000000", label="DLF")
plt.semilogx(freqs, T_phase, color="#9C9C9C", label="T")
plt.semilogx(freqs, TN_phase, color="#707070", label="TN")
plt.semilogx(freqs, H_RL_phase, color="#FF00DD", label="$H_{RL}$")
plt.semilogx(freqs, NTF_REF_phase, color="#FF0000", label="$NTF_{REF}$")
plt.semilogx(freqs, NTFN_BBPD_phase, color="#CCCF02", label="$NTFN_{BBPD}$")
plt.semilogx(freqs, NTF_BBPD_phase, color="#929408", label="$NTF_{BBPD}$")
plt.semilogx(freqs, NTFN_DCO_phase, color="#6F72FF", label="$NTFN_{DCO}$")
plt.semilogx(freqs, NTF_DCO_phase, color="#0000FF", label="$NTF_{DCO}$")
plt.ylabel('Phase (rad)')
plt.xlabel('Frequency (Hz)')
plt.grid(True, which="both", ls="-")
plt.legend(fontsize=8, loc='lower left')

plt.tight_layout()
plt.savefig("outputs/pll_filter.pdf")

plt.show()